# Decision Tree regression

Decision Tree baseline using the IM-Corr inputs.

## 1. Configuration

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import display
from matplotlib import rcParams
from matplotlib.ticker import FuncFormatter, MultipleLocator
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from threadpoolctl import threadpool_limits

get_ipython().run_line_magic("matplotlib", "inline")

DATA_PATH = Path("data/Data_clean_english.xlsx")
TARGET_COLS = ["MIDR", "MFA"]
SPLIT_SEED = 42
ARCH_COLS = [
    "stories",
    "first_story_height_m",
    "typical_story_height_m",
    "span_length_m",
    "column_section_dim1_m",
    "fundamental_period_rcf_s",
    "beam_section_dim2_m",
]
PDYBRB_COLS = [
    "apdybrb_fy2_mean",
    "apdybrb_fy2_std",
    "apdybrb_fy2_max",
    "apdybrb_fy2_min",
    "apdybrb_fy2_sum",
    "apdybrb_fy2_var",
    "first_stage_yield_ratio",
    "brace_arrangement",
    "apdybrb_activation_disp_mean",
    "apdybrb_activation_disp_std",
    "apdybrb_activation_disp_max",
    "apdybrb_activation_disp_min",
    "apdybrb_activation_disp_sum",
    "apdybrb_activation_disp_var",
]
CAT_COLS = ["stories", "brace_arrangement", "first_stage_yield_ratio"]
CATEGORY_MAPS = {
    "stories": {5: 0, 7: 1, 9: 2, 11: 3},
    "brace_arrangement": {1: 0, 2: 1, 3: 2},
    "first_stage_yield_ratio": {0.4: 0, 0.5: 1},
}
IM_SCHEMES = {
    "IM-Corr": [
        "SI (cm)",
        "EPV (cm/s)",
        "VSI (cm)",
        "SAavg (g)",
        "PGV (cm/s)",
        "PSV (cm/s)",
        "Sa (g)",
        "Vrms (cm/s)",
        "PGA (g)",
        "ASI (cm/s)",
        "EPA (g)",
        "PSA (g)",
        "Ia (m/s)",
        "Arms (g)",
    ]
}
MODEL_NAME = "Decision Tree"
RESULTS_ROOT = Path("results")
SAVE_DIR = RESULTS_ROOT / MODEL_NAME
FIG_DIR = SAVE_DIR / "figures"
FIXED_PARAMS = {"max_depth": 15, "min_samples_split": 5, "min_samples_leaf": 2}
RUN_CV = False
PARAM_GRID = {"max_depth": list(range(3, 16, 2))}
LEAF_CANDIDATES = [1, 2, 4, 8]
SPLIT_CANDIDATES = [2, 5, 10, 20]
TUNE_N_JOBS = 1
INTERNAL_CV_FOLDS = 5
MODEL_JOBS = 1
BLAS_THREADS = 1
OMP_THREADS = 1

## 2. Data preparation

In [ ]:
data = pd.read_excel(DATA_PATH)
data.columns = data.columns.astype(str).str.strip()
data = data.loc[~((data["MFA"] > 15) | (data["MIDR"] > 0.2))].reset_index(drop=True)
ratio = pd.to_numeric(
    data["first_stage_yield_ratio"]
    .astype(str)
    .str.strip()
    .str.replace("%", "", regex=False)
)
data["first_stage_yield_ratio"] = np.round(np.where(ratio > 1, ratio / 100, ratio), 6)
for column in ["stories", "brace_arrangement"]:
    data[column] = pd.to_numeric(data[column], errors="raise")

scheme_name = "IM-Corr"
input_cols = ARCH_COLS + PDYBRB_COLS + IM_SCHEMES[scheme_name]
numeric_cols = [column for column in input_cols if column not in CAT_COLS]
numeric = data[numeric_cols].to_numpy(dtype=float)
y_raw = data[TARGET_COLS].to_numpy(dtype=float)
if not np.isfinite(numeric).all() or not np.isfinite(y_raw).all():
    raise ValueError("Inputs or targets contain NaN/Inf.")

onehot_blocks, onehot_names = [], []
for column, mapping in CATEGORY_MAPS.items():
    codes = data[column].map(mapping)
    if codes.isna().any():
        raise ValueError(f"Unknown or missing categories in {column}.")
    onehot_blocks.append(np.eye(len(mapping))[codes.to_numpy(dtype=int)])
    onehot_names.extend(f"{column}={value}" for value in mapping)
onehot = np.column_stack(onehot_blocks)
feature_names = numeric_cols + onehot_names

test2_mask = np.zeros(len(data), dtype=bool)
for story, pga in [(5, 0.5), (7, 0.4), (9, 0.3), (11, 0.2)]:
    test2_mask |= (data["stories"].to_numpy() == story) & np.isclose(
        data["PGA (g)"].to_numpy(), pga, rtol=0, atol=1e-6
    )
train_idx, test1_idx = train_test_split(
    np.flatnonzero(~test2_mask), test_size=0.3, random_state=SPLIT_SEED
)
split_indices = {
    "Train": train_idx,
    "Test1": test1_idx,
    "Test2": np.flatnonzero(test2_mask),
}
scaler_x = StandardScaler().fit(numeric[train_idx])
scaler_y = StandardScaler().fit(y_raw[train_idx])
X = np.column_stack([scaler_x.transform(numeric), onehot])
Y = scaler_y.transform(y_raw)
splits = {
    name: {"X": X[ids], "Y": Y[ids], "y_raw": y_raw[ids], "sample_id": ids}
    for name, ids in split_indices.items()
}

## 3. Training and prediction

In [ ]:
estimator = DecisionTreeRegressor(random_state=42)
estimator.set_params(**FIXED_PARAMS)
cv_results = pd.DataFrame()
selection_X = np.column_stack([numeric, onehot])[train_idx]
selection_pipeline = Pipeline(
    [
        (
            "scale",
            ColumnTransformer(
                [("numeric", StandardScaler(), list(range(len(numeric_cols))))],
                remainder="passthrough",
                sparse_threshold=0,
            ),
        ),
        ("dtype", FunctionTransformer(np.asarray, kw_args={"dtype": np.float64})),
        ("model", clone(estimator)),
    ]
)
with (
    threadpool_limits(limits=BLAS_THREADS, user_api="blas"),
    threadpool_limits(limits=OMP_THREADS, user_api="openmp"),
):
    training_started = time.perf_counter()
    if RUN_CV:
        search = GridSearchCV(
            selection_pipeline,
            {f"model__{key}": values for key, values in PARAM_GRID.items()},
            cv=KFold(n_splits=INTERNAL_CV_FOLDS, shuffle=False),
            n_jobs=TUNE_N_JOBS,
            scoring="neg_mean_squared_error",
            refit=False,
            error_score="raise",
        )
        search.fit(selection_X, splits["Train"]["Y"])
        estimator.set_params(
            **{
                key.removeprefix("model__"): value
                for key, value in search.best_params_.items()
            }
        )
        cv_results = pd.DataFrame(search.cv_results_)
    selection_pipeline.set_params(model=clone(estimator))
    inner_cv = KFold(n_splits=INTERNAL_CV_FOLDS, shuffle=False)
    internal_grid, stopping_rules = [], set()
    for leaf in LEAF_CANDIDATES:
        for split in SPLIT_CANDIDATES:
            rule = (leaf, max(split, 2 * leaf))
            if rule not in stopping_rules:
                stopping_rules.add(rule)
                internal_grid.append(
                    {
                        "model__min_samples_leaf": [leaf],
                        "model__min_samples_split": [split],
                    }
                )
    internal_search = GridSearchCV(
        selection_pipeline,
        internal_grid,
        cv=inner_cv,
        n_jobs=TUNE_N_JOBS,
        scoring="neg_mean_squared_error",
        refit=False,
        error_score="raise",
    )
    internal_search.fit(selection_X, splits["Train"]["Y"])
    selected_parameters = {
        key.removeprefix("model__"): int(value)
        for key, value in internal_search.best_params_.items()
    }
    estimator.set_params(**selected_parameters)
    selection_results = pd.DataFrame(internal_search.cv_results_)
    validation_fold = np.empty(len(train_idx), dtype=int)
    for fold, (_, validation_positions) in enumerate(inner_cv.split(selection_X), 1):
        validation_fold[validation_positions] = fold
    selection_manifest = pd.DataFrame(
        {"sample_id": train_idx, "validation_fold": validation_fold}
    )
    estimator.fit(splits["Train"]["X"], splits["Train"]["Y"])
    training_seconds = time.perf_counter() - training_started

    predictions, prediction_times, metric_rows = {}, {}, []
    for name in ["Test1", "Test2"]:
        arrays = splits[name]
        started = time.perf_counter()
        scaled_prediction = estimator.predict(arrays["X"])
        prediction_times[name] = time.perf_counter() - started
        predicted = scaler_y.inverse_transform(scaled_prediction)
        table = pd.DataFrame({"sample_id": arrays["sample_id"]})
        for j, target in enumerate(TARGET_COLS):
            truth, estimate = arrays["y_raw"][:, j], predicted[:, j]
            table[f"{target}_true"] = truth
            table[f"{target}_pred"] = estimate
            metric_rows.append(
                {
                    "Split": name,
                    "Target": target,
                    "R2": r2_score(truth, estimate),
                    "RMSE": np.sqrt(mean_squared_error(truth, estimate)),
                    "MAE": mean_absolute_error(truth, estimate),
                }
            )
        predictions[name] = table


metrics = pd.DataFrame(metric_rows)
timing = pd.DataFrame(
    [
        {
            "Train_Time_s": training_seconds,
            "Pred_Time_Test1_s": prediction_times["Test1"],
            "Pred_Time_Test2_s": prediction_times["Test2"],
        }
    ]
)
display(metrics, timing)

In [ ]:
selection_results = selection_results.drop(
    columns=[column for column in selection_results if column.endswith("_time")]
)
cv_results = cv_results.drop(
    columns=[column for column in cv_results if column.endswith("_time")]
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
for name, table in predictions.items():
    table.to_excel(
        SAVE_DIR / f"{name.lower()}_predictions(Decision Tree)_{scheme_name}.xlsx",
        index=False,
    )
with pd.ExcelWriter(SAVE_DIR / "metrics_summary(Decision Tree).xlsx") as writer:
    metrics.to_excel(writer, sheet_name="Metrics", index=False)
    timing.to_excel(writer, sheet_name="Timing", index=False)
    selection_results.to_excel(writer, sheet_name="Internal_Selection", index=False)
    if RUN_CV:
        cv_results.to_excel(writer, sheet_name="External_CV", index=False)
selection_manifest.to_csv(SAVE_DIR / "internal_selection_splits.csv", index=False)

## 4. Prediction figures

In [ ]:
rcParams["font.family"] = "serif"
rcParams["font.serif"] = ["Times New Roman", "DejaVu Serif"]
rcParams["mathtext.fontset"] = "stix"
rcParams["axes.unicode_minus"] = False
for target, limit, step in [("MIDR", 0.09, 0.02), ("MFA", 20, 5)]:
    fig, ax = plt.subplots(figsize=(7, 6))
    for name, color, marker in [("Test1", "green", "o"), ("Test2", "red", "s")]:
        table = predictions[name]
        score = r2_score(table[f"{target}_true"], table[f"{target}_pred"])
        ax.scatter(
            table[f"{target}_true"],
            table[f"{target}_pred"],
            c=color,
            marker=marker,
            s=40,
            alpha=0.6,
            edgecolors="k",
            linewidths=0.5,
            label=rf"{name}: $R^2$ = {score:.3f}",
        )
    ax.plot([0, limit], [0, limit], color="black", linewidth=1.5)
    ax.set(xlim=(0, limit), ylim=(0, limit))
    ax.xaxis.set_major_locator(MultipleLocator(step))
    ax.yaxis.set_major_locator(MultipleLocator(step))
    for axis in (ax.xaxis, ax.yaxis):
        axis.set_major_formatter(
            FuncFormatter(lambda value, _: "" if abs(value) < 1e-12 else f"{value:g}")
        )
    ax.annotate(
        "0",
        xy=(0, 0),
        xycoords="axes fraction",
        xytext=(-7, -7),
        textcoords="offset points",
        ha="right",
        va="top",
        fontsize=20,
        annotation_clip=False,
    )
    ax.set_xlabel(f"Reference value ({target})", fontsize=22)
    ax.set_ylabel(f"Predicted value ({target})", fontsize=22)
    ax.tick_params(labelsize=20, direction="in", top=True, right=True, length=5)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="upper left", fontsize=18, framealpha=0.9, edgecolor="gray")
    fig.tight_layout()
    for extension in ["png", "pdf"]:
        fig.savefig(
            FIG_DIR / f"Decision Tree_Comparison_{target}_{scheme_name}.{extension}",
            dpi=300,
        )
    display(fig)
    plt.close(fig)